# CS1 EXP-2 -- SVD(TF-IDF) + Static Features -> MLP (rotating 5-fold, staged nested search)

## 1. Workspace root

In [1]:
from pathlib import Path

WORKSPACE_ROOT = Path.cwd()

print("WORKSPACE_ROOT:", WORKSPACE_ROOT)


WORKSPACE_ROOT: /workspace


## 2. Runtime switches

Run profile first. After the profile succeeds, disable profile and enable the official development CV.


In [2]:
RUN_PROFILE_FOLD = False
RUN_OFFICIAL = True
REQUIRE_GPU_FOR_OFFICIAL_RUN = True

REPO_URL = "https://github.com/EnomisLP/DiverseVul--IS-Project.git"
REPO_BRANCH = "Recovery"
MODEL_RANDOM_STATE = 42

CODE_COLUMN = "normalized_code"
#CODE_COLUMN = "abstracted_code_v1"
CODE_COLUMN_TAG = "abstracted" if CODE_COLUMN == "abstracted_code_v1" else "normalized"

print("RUN_PROFILE_FOLD:", RUN_PROFILE_FOLD)
print("RUN_OFFICIAL:", RUN_OFFICIAL)
print("CODE_COLUMN:", CODE_COLUMN)


RUN_PROFILE_FOLD: False
RUN_OFFICIAL: True
CODE_COLUMN: normalized_code


## 3. Define paths


In [3]:
from pathlib import Path

DRIVE_ROOT = WORKSPACE_ROOT / "IntelligentSystemProject" / "VulnerabilityDetectionData"
PROCESSED_DIR = DRIVE_ROOT / "processed"
MANIFEST_ROOT = DRIVE_ROOT / "manifests"
OUTPUT_ROOT = DRIVE_ROOT / "outputs"

DOWNSAMPLED_PARQUET = PROCESSED_DIR / "rdiversevul_cs1_normalized_plus_abstracted_v2_downsampled20k.parquet"
MANIFEST_PATH = MANIFEST_ROOT / "cs1_shared_rotating_5fold_v1" / "project_grouped_5fold_manifest.parquet"
STATIC_FEATURE_DIR = PROCESSED_DIR / "static_features_downsampled20k"
STATIC_FEATURE_PATH = STATIC_FEATURE_DIR / "cs1_static_features_v1.parquet"

EXP2_OUTPUT_DIR = OUTPUT_ROOT / f"exp2_mlp_rotating5fold_{CODE_COLUMN_TAG}"

for directory in [PROCESSED_DIR, MANIFEST_ROOT, OUTPUT_ROOT, STATIC_FEATURE_DIR, EXP2_OUTPUT_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

print("Downsampled parquet:", DOWNSAMPLED_PARQUET)
print("Manifest:", MANIFEST_PATH)
print("Static feature cache:", STATIC_FEATURE_PATH)
print("Output dir:", EXP2_OUTPUT_DIR)


Downsampled parquet: /workspace/IntelligentSystemProject/VulnerabilityDetectionData/processed/rdiversevul_cs1_normalized_plus_abstracted_v2_downsampled20k.parquet
Manifest: /workspace/IntelligentSystemProject/VulnerabilityDetectionData/manifests/cs1_shared_rotating_5fold_v1/project_grouped_5fold_manifest.parquet
Static feature cache: /workspace/IntelligentSystemProject/VulnerabilityDetectionData/processed/static_features_downsampled20k/cs1_static_features_v1.parquet
Output dir: /workspace/IntelligentSystemProject/VulnerabilityDetectionData/outputs/exp2_mlp_rotating5fold_normalized


## 4. Clone or refresh the repository branch


In [4]:
from pathlib import Path
import sys

REPO_DIR = WORKSPACE_ROOT / "DiverseVul--IS-Project"
PROJECT_DIR = REPO_DIR / "vuln-detection"
SRC_DIR = PROJECT_DIR / "src"

import urllib.request
import zipfile


def download_and_extract_repo(repo_url, branch, target_dir):
    if target_dir.exists():
        print(f"Repository already exists at {target_dir}")
        return

    print(f"Downloading {repo_url} (branch: {branch}) without git...")
    clean_url = repo_url.removesuffix(".git")
    zip_url = f"{clean_url}/archive/refs/heads/{branch}.zip"
    target_dir.parent.mkdir(parents=True, exist_ok=True)
    zip_path = target_dir.parent / f"{target_dir.name}_download_temp.zip"

    urllib.request.urlretrieve(zip_url, zip_path)

    print("Extracting files...")
    with zipfile.ZipFile(zip_path, "r") as zip_ref:
        zip_ref.extractall(target_dir.parent)

    repo_name = clean_url.split("/")[-1]
    extracted_folder = target_dir.parent / f"{repo_name}-{branch}"
    if extracted_folder.exists():
        extracted_folder.rename(target_dir)

    zip_path.unlink()
    print(f"Repository ready at {target_dir}")

download_and_extract_repo(REPO_URL, REPO_BRANCH, REPO_DIR)

if not SRC_DIR.exists():
    raise FileNotFoundError(f"Expected source directory does not exist: {SRC_DIR}")

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

print("Repository:", REPO_DIR)
print("Source path:", SRC_DIR)


Repository already exists at /workspace/DiverseVul--IS-Project
Repository: /workspace/DiverseVul--IS-Project
Source path: /workspace/DiverseVul--IS-Project/vuln-detection/src


## 5. Verify required repository files


In [5]:
required_repo_files = [
    SRC_DIR / "utils" / "evaluation.py",
    SRC_DIR / "utils" / "split_manifest.py",
    SRC_DIR / "case_study_1" / "static_features.py",
    SRC_DIR / "case_study_1" / "exp2" / "exp2_mlp.py",
]

missing_repo_files = [str(path) for path in required_repo_files if not path.exists()]
if missing_repo_files:
    raise FileNotFoundError("Required EXP-2 files are missing:\n" + "\n".join(missing_repo_files))

print("Required Case Study 1 files are present.")


Required Case Study 1 files are present.


## 6. Install dependencies


In [6]:
import sys
import subprocess

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "numpy",
        "pandas",
        "scipy",
        "scikit-learn",
        "matplotlib",
        "pyyaml",
        "pyarrow",
        "joblib",
    ],
    check=True,
)
print("Dependencies installed or already available.")


Dependencies installed or already available.


## 7. Import modules and check accelerator


In [7]:
import importlib
import json
import os
import subprocess
import time

os.environ["CUDA_VISIBLE_DEVICES"] = "0"

import numpy as np
import pandas as pd
import torch
from IPython.display import display

exp2_mlp = importlib.import_module("case_study_1.exp2.exp2_mlp")
static_features = importlib.import_module("case_study_1.static_features")
split_manifest = importlib.import_module("utils.split_manifest")
evaluation = importlib.import_module("utils.evaluation")

required_exp2_api = ["EXP2_VERSION", "Exp2Config", "run_exp2_profile_fold", "run_exp2"]
missing_exp2_api = [name for name in required_exp2_api if not hasattr(exp2_mlp, name)]
if missing_exp2_api:
    raise AttributeError(f"EXP-2 runner is missing API: {missing_exp2_api}")

print("EXP-2 runner:", exp2_mlp.__file__)
print("EXP-2 version:", exp2_mlp.EXP2_VERSION)
print("Static feature count:", len(static_features.FEATURE_COLUMNS))
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    assert torch.cuda.device_count() == 1, f"Expected 1 visible GPU, but PyTorch sees {torch.cuda.device_count()}"
    print("Locked to single GPU:", torch.cuda.get_device_name(0))


EXP-2 runner: /workspace/DiverseVul--IS-Project/vuln-detection/src/case_study_1/exp2/exp2_mlp.py
EXP-2 version: cs1-exp2-svd-static-mlp-v2-nested-rotating-5fold
Static feature count: 54
PyTorch: 2.5.1+cu124
CUDA available: True
Locked to single GPU: NVIDIA A100-SXM4-80GB


## 8. Load the downsampled dataset, shared 5-fold manifest, and static features

In [8]:
if not DOWNSAMPLED_PARQUET.is_file():
    raise FileNotFoundError(
        f"Missing downsampled parquet: {DOWNSAMPLED_PARQUET}\n"
        "Run notebooks/scope2_preprocessing.ipynb first to build it."
    )
if not MANIFEST_PATH.is_file():
    raise FileNotFoundError(
        f"Missing manifest: {MANIFEST_PATH}\n"
        "Run notebooks/scope2_preprocessing.ipynb's manifest-generation section first."
    )

full_df = pd.read_parquet(DOWNSAMPLED_PARQUET)
manifest_df = split_manifest.load_manifest(MANIFEST_PATH, config=split_manifest.SplitConfig(n_splits=5, random_state=42))

required_full_columns = {"source_row_id", "code", "normalized_code", "abstracted_code_v1", "label", "project"}
missing_full_columns = required_full_columns.difference(full_df.columns)
if missing_full_columns:
    raise KeyError(f"Downsampled dataset missing columns: {sorted(missing_full_columns)}")

static_df = None
if STATIC_FEATURE_PATH.is_file():
    cached_static_df = pd.read_parquet(STATIC_FEATURE_PATH)
    if set(cached_static_df["source_row_id"]) == set(full_df["source_row_id"]):
        static_df = cached_static_df
        print("Loaded cached static features:", STATIC_FEATURE_PATH)
    else:
        # The cache is keyed by a fixed filename, not by which downsampled
        # dataset produced it; if the downsample is ever regenerated, a stale
        # cache from an older row set would otherwise pass the .is_file()
        # check and only fail much later. Recompute instead of trusting it.
        print("Cached static features do not match the current downsampled dataset; recomputing.")
    del cached_static_df

if static_df is None:
    print("Computing static features for the downsampled dataset (deterministic, no leakage)...")
    static_config = static_features.StaticFeatureConfig(source_id_column="source_row_id", code_column="code")
    static_df = static_features.extract_static_feature_frame(full_df, config=static_config)
    static_features.save_static_feature_artifacts(
        static_df, STATIC_FEATURE_DIR, config=static_config, source_dataset_path=DOWNSAMPLED_PARQUET
    )
    print("Saved static features to:", STATIC_FEATURE_PATH)

print("Downsampled dataset:", full_df.shape)
print("Manifest:", manifest_df.shape)
print("Static features:", static_df.shape)

fold_summary_diag = split_manifest.summarize_manifest(
    manifest_df, config=split_manifest.SplitConfig(n_splits=5, random_state=42)
)
display(fold_summary_diag)

fold_size_ratio = fold_summary_diag["test_rows"].max() / fold_summary_diag["test_rows"].min()
print(f"Fold test-size balance: smallest={fold_summary_diag['test_rows'].min()} rows, "
      f"largest={fold_summary_diag['test_rows'].max()} rows, ratio={fold_size_ratio:.2f}x")
if fold_size_ratio > 2.0:
    print("WARNING: fold sizes are notably imbalanced (ratio > 2x) -- "
          "regenerate the manifest via scope2_preprocessing.ipynb with an updated "
          "DOWNSAMPLE_MAX_ROWS_PER_PROJECT.")
else:
    print("Fold sizes look reasonably balanced.")


Loaded cached static features: /workspace/IntelligentSystemProject/VulnerabilityDetectionData/processed/static_features_downsampled20k/cs1_static_features_v1.parquet
Downsampled dataset: (20000, 7)
Manifest: (20000, 4)
Static features: (20000, 55)


,fold,test_rows,test_vulnerable,test_non_vulnerable,test_positive_rate,positive_rate_delta_from_global,test_unique_projects,train_rows,train_unique_projects,train_test_project_overlap,test_row_share,test_project_share
0,0,3473,198,3275,0.057011,-0.002039,151,16527,615,0,0.17365,0.197128
1,1,4383,268,4115,0.061145,0.002095,156,15617,610,0,0.21915,0.203655
2,2,4341,224,4117,0.051601,-0.007449,156,15659,610,0,0.21705,0.203655
3,3,4092,253,3839,0.061828,0.002778,151,15908,615,0,0.20460,0.197128
4,4,3711,238,3473,0.064134,0.005084,152,16289,614,0,0.18555,0.198433


Fold test-size balance: smallest=3473 rows, largest=4383 rows, ratio=1.26x
Fold sizes look reasonably balanced.


## 9. Manifest validation and empty-code guard

In [9]:
if full_df["source_row_id"].duplicated().any():
    raise RuntimeError("full_df contains duplicate source_row_id values.")
if manifest_df["source_row_id"].duplicated().any():
    raise RuntimeError("manifest_df contains duplicate source_row_id values.")
if static_df["source_row_id"].duplicated().any():
    raise RuntimeError("static_df contains duplicate source_row_id values.")

full_indexed = full_df.set_index("source_row_id", drop=False)
manifest_ids = set(manifest_df["source_row_id"].tolist())
if manifest_ids != set(full_indexed.index):
    raise RuntimeError(
        "Manifest coverage does not match the downsampled dataset exactly. "
        f"Missing={len(set(full_indexed.index) - manifest_ids)}, extra={len(manifest_ids - set(full_indexed.index))}"
    )

joined = manifest_df.set_index("source_row_id").join(full_indexed[["label", "project"]], rsuffix="_dataset")
if not (joined["label"].astype(int) == joined["label_dataset"].astype(int)).all():
    raise RuntimeError("Label mismatch between downsampled dataset and manifest.")
if not (joined["project"].astype(str) == joined["project_dataset"].astype(str)).all():
    raise RuntimeError("Project mismatch between downsampled dataset and manifest.")

static_ids = set(static_df["source_row_id"].tolist())
if static_ids != manifest_ids:
    raise RuntimeError(
        "Static feature cache coverage does not match the dataset exactly. "
        f"Missing={len(manifest_ids - static_ids)}, extra={len(static_ids - manifest_ids)}"
    )

dataset_frame = full_indexed.loc[list(manifest_ids)].copy().reset_index(drop=True)
dataset_frame = dataset_frame[
    ["source_row_id", "code", "abstracted_code_v1", "normalized_code", "label", "project"]
].copy()

print("Dataset ready:", dataset_frame.shape, "| positive rate:", dataset_frame["label"].mean())
print("Unique projects:", dataset_frame["project"].nunique())


Dataset ready: (20000, 6) | positive rate: 0.05905
Unique projects: 766


## 10. Empty-code guard

In [10]:
EMPTY_CODE_SENTINEL = "EMPTY_ABSTRACTED_CODE_SAMPLE"
dataset_frame[CODE_COLUMN] = dataset_frame[CODE_COLUMN].fillna("").astype(str)
empty_mask = dataset_frame[CODE_COLUMN].str.strip().eq("")
n_empty = int(empty_mask.sum())
if n_empty:
    dataset_frame.loc[empty_mask, CODE_COLUMN] = EMPTY_CODE_SENTINEL
    print(f"Replaced {n_empty} empty {CODE_COLUMN} rows with sentinel token.")
assert not dataset_frame[CODE_COLUMN].str.strip().eq("").any()
print("Empty-code guard passed.")


Empty-code guard passed.


## 11. Configure EXP-2

In [11]:
config = exp2_mlp.Exp2Config(
    experiment_name=f"cs1_exp2_mlp_{CODE_COLUMN_TAG}",
    code_column=CODE_COLUMN,
    source_id_column="source_row_id",
    label_column="label",
    project_column="project",
    fold_column="fold",
    n_splits=5,
    random_state=MODEL_RANDOM_STATE,
    device="cuda" if torch.cuda.is_available() else "cpu",
    verbose=True,
)

print("Code column:", config.code_column)
print("word_max_features_grid:", config.word_max_features_grid)
print("char_max_features_grid:", config.char_max_features_grid)
print("svd_n_components_grid:", config.svd_n_components_grid)
print("hidden_dim_1_grid:", config.hidden_dim_1_grid)
print("hidden_dim_2_grid:", config.hidden_dim_2_grid)
print("learning_rate_grid:", config.learning_rate_grid)
print("Device:", config.device)
print("Output directory:", EXP2_OUTPUT_DIR)


Code column: normalized_code
word_max_features_grid: (20000, 50000, 80000)
char_max_features_grid: (30000, 60000, 90000)
svd_n_components_grid: (128, 256, 384)
hidden_dim_1_grid: (64, 128, 256)
hidden_dim_2_grid: (32, 64, 128)
learning_rate_grid: (0.001, 0.0005, 0.0001)
Device: cuda
Output directory: /workspace/IntelligentSystemProject/VulnerabilityDetectionData/outputs/exp2_mlp_rotating5fold_normalized


## 12. Optional profile run -- staged inner search + refit for one outer fold

In [12]:
if RUN_PROFILE_FOLD:
    profile_start = time.perf_counter()
    profile = exp2_mlp.run_exp2_profile_fold(
        normalized_frame=dataset_frame,
        static_features_frame=static_df,
        manifest=manifest_df,
        fold_id=4,
        config=config,
    )
    print("Profile duration minutes:", (time.perf_counter() - profile_start) / 60)
    print("Selected hyperparameters:")
    display(pd.DataFrame([profile["selection"]]))
    print("Fold metrics:")
    display(profile["profile_metrics"])
else:
    print("RUN_PROFILE_FOLD=False; skipping profile.")


RUN_PROFILE_FOLD=False; skipping profile.


## 13. Inspect profile result

In [13]:
if "profile" not in globals():
    print("No profile result in memory. Run the profile cell first or skip this section.")
else:
    print("Inner search summary (Stage A then Stage B):")
    display(profile["inner_search"])
    print("\nFold training metadata:")
    display(profile["training_metadata"])


No profile result in memory. Run the profile cell first or skip this section.


## 14. Official rotating 5-fold run

In [14]:
def _resolve_repo_commit(repo_root: Path, repo_branch: str) -> str:
    try:
        return subprocess.check_output(
            ["git", "-C", str(repo_root), "rev-parse", "HEAD"], stderr=subprocess.DEVNULL,
        ).decode().strip()
    except (FileNotFoundError, subprocess.CalledProcessError):
        return f"unknown (repo fetched via zip archive, branch={repo_branch}, no .git metadata)"


if RUN_OFFICIAL:
    if REQUIRE_GPU_FOR_OFFICIAL_RUN and not torch.cuda.is_available():
        raise RuntimeError(
            "GPU is required for this official MLP run, but torch.cuda.is_available() is False. "
            "Set REQUIRE_GPU_FOR_OFFICIAL_RUN=False in the runtime-switches cell to accept a much slower CPU run."
        )

    results = exp2_mlp.run_exp2(
        normalized_frame=dataset_frame,
        static_features_frame=static_df,
        manifest=manifest_df,
        config=config,
        output_dir=EXP2_OUTPUT_DIR,
        additional_metadata={
            "input_parquet": str(DOWNSAMPLED_PARQUET),
            "input_column": CODE_COLUMN,
            "manifest_path": str(MANIFEST_PATH),
            "static_feature_cache_path": str(STATIC_FEATURE_PATH),
            "repo_commit": _resolve_repo_commit(REPO_DIR, REPO_BRANCH),
        },
    )
    print("\nOfficial EXP-2 run complete.")
else:
    results = None
    print("RUN_OFFICIAL=False; official training skipped.")


[09:30:30] CS1-EXP2 official run started: 5 rotating outer folds.
[09:30:30] Outer fold 1/5 | Stage A (representation) search started.
[13:06:09] Outer fold 1/5 | Stage B (architecture) search started.
[13:14:34] Outer fold 1/5 | selected max_features=(80000,90000) min_df=(2,8) svd=128 hidden=(128,64) lr=0.001 threshold=0.66 (inner PR-AUC=0.1687).
[13:15:30] Outer fold 1/5 | fit+scored in 56.1s.
[13:15:30] Outer fold 2/5 | Stage A (representation) search started.
[16:35:18] Outer fold 2/5 | Stage B (architecture) search started.
[16:42:50] Outer fold 2/5 | selected max_features=(80000,60000) min_df=(3,8) svd=256 hidden=(256,32) lr=0.0001 threshold=0.58 (inner PR-AUC=0.1615).
[16:43:49] Outer fold 2/5 | fit+scored in 59.1s.
[16:43:49] Outer fold 3/5 | Stage A (representation) search started.
[19:56:23] Outer fold 3/5 | Stage B (architecture) search started.
[20:04:18] Outer fold 3/5 | selected max_features=(80000,60000) min_df=(2,8) svd=256 hidden=(64,32) lr=0.0005 threshold=0.62 (inner

## 15. Display result summary

In [15]:
if results is None:
    print("No official results object in memory. Set RUN_OFFICIAL=True.")
else:
    pooled = results["evaluation"]["pooled_metrics"]
    fold_metrics = results["evaluation"]["fold_metrics"]
    fold_summary = results["evaluation"]["fold_summary"]
    fold_training = results["fold_training"]

    print("Pooled OOF metrics (secondary cross-check):")
    display(pd.DataFrame([{"metric": k, "value": v} for k, v in pooled.items()]))

    print("Per-fold metrics:")
    display(fold_metrics)

    print("Mean +/- std across the 5 outer folds (headline result):")
    display(fold_summary)

    print("Selected hyperparameters by outer fold:")
    display(fold_training[[
        "fold", "word_max_features", "char_max_features", "word_min_df", "char_min_df",
        "svd_n_components", "hidden_dim_1", "hidden_dim_2", "learning_rate", "decision_threshold",
    ]])

    print("Artifacts:", results.get("artifacts"))


Pooled OOF metrics (secondary cross-check):


,metric,value
0,n_samples,20000.000000
1,vulnerable_1,1181.000000
2,non_vulnerable_0,18819.000000
3,positive_rate,0.059050
4,threshold,0.602000
5,average_precision_pr_auc,0.154511
6,precision,0.175834
7,recall,0.330229
8,f1,0.229479
9,mcc,0.174977


Per-fold metrics:


,fold,n_samples,vulnerable_1,non_vulnerable_0,positive_rate,threshold,average_precision_pr_auc,precision,recall,f1,...,negative_predictive_value,false_positive_rate,false_negative_rate,true_negative,false_positive,false_negative,true_positive,predicted_positive,predicted_positive_rate,test_unique_projects
0,0,3473,198,3275,0.057011,0.602,0.138085,0.145558,0.388889,0.211829,...,0.958899,0.138015,0.611111,2823,452,121,77,529,0.152318,151
1,1,4383,268,4115,0.061145,0.602,0.153523,0.182336,0.238806,0.206785,...,0.949405,0.069745,0.761194,3828,287,204,64,351,0.080082,156
2,2,4341,224,4117,0.051601,0.602,0.158941,0.161290,0.468750,0.240000,...,0.967751,0.132621,0.531250,3571,546,119,105,651,0.149965,156
3,3,4092,253,3839,0.061828,0.602,0.165423,0.218232,0.312253,0.256911,...,0.953351,0.073717,0.687747,3556,283,174,79,362,0.088465,151
4,4,3711,238,3473,0.064134,0.602,0.191022,0.200000,0.273109,0.230906,...,0.948907,0.074863,0.726891,3213,260,173,65,325,0.087577,152


Mean +/- std across the 5 outer folds (headline result):


,metric,mean,std,min,max
0,positive_rate,0.059144,0.004938,0.051601,0.064134
1,threshold,0.602000,0.000000,0.602000,0.602000
2,average_precision_pr_auc,0.161399,0.019399,0.138085,0.191022
3,precision,0.181483,0.029122,0.145558,0.218232
4,recall,0.336361,0.092716,0.238806,0.468750
5,f1,0.229286,0.020564,0.206785,0.256911
6,mcc,0.178702,0.025640,0.149239,0.208265
7,accuracy,0.868287,0.025413,0.835013,0.888319
8,balanced_accuracy,0.619285,0.031748,0.584531,0.668065
9,specificity,0.902208,0.034362,0.861985,0.930255


Selected hyperparameters by outer fold:


,fold,word_max_features,char_max_features,word_min_df,char_min_df,svd_n_components,hidden_dim_1,hidden_dim_2,learning_rate,decision_threshold
0,0,80000,90000,2,8,128,128,64,0.0010,0.66
1,1,80000,60000,3,8,256,256,32,0.0001,0.58
2,2,80000,60000,2,8,256,64,32,0.0005,0.62
3,3,50000,60000,3,8,384,256,32,0.0005,0.59
4,4,80000,60000,2,5,384,128,32,0.0005,0.56


Artifacts: Exp2Artifacts(config_json=PosixPath('/workspace/IntelligentSystemProject/VulnerabilityDetectionData/outputs/exp2_mlp_rotating5fold_normalized/cs1_exp2_mlp_normalized_config.json'), fold_training_csv=PosixPath('/workspace/IntelligentSystemProject/VulnerabilityDetectionData/outputs/exp2_mlp_rotating5fold_normalized/cs1_exp2_mlp_normalized_fold_training.csv'), inner_search_csv=PosixPath('/workspace/IntelligentSystemProject/VulnerabilityDetectionData/outputs/exp2_mlp_rotating5fold_normalized/cs1_exp2_mlp_normalized_inner_search.csv'), training_history_csv=PosixPath('/workspace/IntelligentSystemProject/VulnerabilityDetectionData/outputs/exp2_mlp_rotating5fold_normalized/cs1_exp2_mlp_normalized_training_history.csv'), run_metadata_json=PosixPath('/workspace/IntelligentSystemProject/VulnerabilityDetectionData/outputs/exp2_mlp_rotating5fold_normalized/cs1_exp2_mlp_normalized_run_metadata.json'), evaluation_paths=EvaluationPaths(predictions_parquet=PosixPath('/workspace/IntelligentSy

## 15b. Confidence interval on pooled OOF PR-AUC (ad hoc)

In [16]:
import utils.confidence_intervals as confidence_intervals

exp2_oof_ci = confidence_intervals.bootstrap_metric_ci(
    results["oof_predictions"],
    metric="average_precision_pr_auc",
    n_bootstrap=1000,
    random_state=42,
)
print(confidence_intervals.format_ci_report(exp2_oof_ci))


average_precision_pr_auc: point estimate = 0.1545
  95% CI (project-block bootstrap): [0.1365, 0.1735]
  valid resamples: 1000/1000 (0 degenerate, dropped)
  n_projects: 766, random_state=42
  Reflects sampling variability within this dataset only; not an estimate of generalization to C functions outside this collection.


## 16. Error analysis: pooled OOF false positives/negatives

In [17]:
oof = results["oof_predictions"].merge(dataset_frame[["source_row_id", "code"]], on="source_row_id", how="left")

false_positives = oof[(oof["label"] == 0) & (oof["y_pred"] == 1)]
false_negatives = oof[(oof["label"] == 1) & (oof["y_pred"] == 0)]

print(f"Extracted {len(false_positives)} False Positives and {len(false_negatives)} False Negatives (pooled OOF, full dataset).")

sample_columns = ["source_row_id", "project", "y_score", "code"]
false_positives[sample_columns].sample(n=min(5, len(false_positives)), random_state=42).to_csv(
    EXP2_OUTPUT_DIR / "sample_false_positives.csv", index=False
)
false_negatives[sample_columns].sample(n=min(5, len(false_negatives)), random_state=42).to_csv(
    EXP2_OUTPUT_DIR / "sample_false_negatives.csv", index=False
)


Extracted 1828 False Positives and 791 False Negatives (pooled OOF, full dataset).
